# Practical Data Cleansing & Conversion (Hands-on Notebook)

This notebook is designed as a **step-by-step workshop** for learners.

You will practice:
1. Filtering and converting data types (CSV/DOCX -> JSONL)
2. Cleansing and normalization of A/S (After-Sales) logs
3. Deduplication techniques (exact + near duplicates)
4. PII filtering and masking
5. Exporting a final training-ready JSONL file

> Run this notebook from top to bottom, one cell at a time.

## 0) Environment Setup

This workshop uses common Python data tools:
- `pandas` for tabular transformations
- `python-docx` for reading/writing DOCX
- `json`, `re`, and `pathlib` for normalization and export

If you do not have these packages, uncomment and run the install command below.

In [1]:
# Uncomment if needed:
%pip install pandas python-docx requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
from pathlib import Path
from datetime import datetime

import pandas as pd

try:
    from docx import Document
except ImportError as exc:
    raise ImportError("python-docx is required. Install with: pip install python-docx") from exc

pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path(".").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

for p in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Base dir:", BASE_DIR)
print("Created:", RAW_DIR, INTERIM_DIR, PROCESSED_DIR, sep="\n")

Base dir: /home/ethan/newgen/KMOU_Course/Module_4
Created:
/home/ethan/newgen/KMOU_Course/Module_4/data/raw
/home/ethan/newgen/KMOU_Course/Module_4/data/interim
/home/ethan/newgen/KMOU_Course/Module_4/data/processed


## 1) Get a Sample Dataset (A/S-like Log Data)

We download a public **Apache combined (access) log** sample (lines your browser leaves on a web server), then **append a few synthetic A/S-style lines** so later steps can extract `ticket=` and `/service/...` paths.

If internet access is unavailable, the notebook will write only the synthetic fallback lines.


In [3]:
import requests

# Combined (access) log format — loghub's Apache_2k.log is error-log style and will not match the Step 2 parser.
sample_log_url = (
    "https://raw.githubusercontent.com/elastic/examples/master/"
    "Common%20Data%20Formats/apache_logs/apache_logs"
)
raw_log_path = RAW_DIR / "apache_sample.log"

fallback_lines = [
    '203.0.113.10 - - [15/Apr/2026:08:31:20 +0000] "GET /service/aircon?ticket=AS-1001 HTTP/1.1" 200 482 "-" "Mozilla/5.0"',
    '198.51.100.77 - - [15/Apr/2026:08:31:30 +0000] "POST /service/washer?ticket=AS-1002 HTTP/1.1" 500 1210 "-" "Mozilla/5.0"',
    '198.51.100.77 - - [15/Apr/2026:08:31:31 +0000] "POST /service/washer?ticket=AS-1002 HTTP/1.1" 500 1210 "-" "Mozilla/5.0"',
    '192.0.2.20 - - [15/Apr/2026:08:33:40 +0000] "GET /service/fridge?ticket=AS-1003 HTTP/1.1" 404 532 "-" "Mozilla/5.0"',
    '192.0.2.20 - - [15/Apr/2026:08:34:01 +0000] "GET /service/fridge?ticket=AS-1003 HTTP/1.1" 200 900 "-" "Mozilla/5.0"',
    '198.51.100.90 - - [15/Apr/2026:09:00:00 +0000] "GET /service/aircon?ticket=AS-1004 HTTP/1.1" 200 640 "-" "Mozilla/5.0"',
]

download_ok = False
try:
    response = requests.get(sample_log_url, timeout=30)
    response.raise_for_status()
    combined_body = response.text.rstrip("\n")
    supplement = "\n".join(fallback_lines)
    raw_log_path.write_text(combined_body + "\n" + supplement + "\n", encoding="utf-8")
    download_ok = True
except Exception as e:
    raw_log_path.write_text("\n".join(fallback_lines), encoding="utf-8")
    print("Download failed -> using fallback synthetic logs.")
    print("Reason:", e)

print("Saved log file:", raw_log_path)
print("Used public dataset:", download_ok)
print("Line count:", len(raw_log_path.read_text(encoding='utf-8', errors='ignore').splitlines()))

Saved log file: /home/ethan/newgen/KMOU_Course/Module_4/data/raw/apache_sample.log
Used public dataset: True
Line count: 10006


## 2) Parse Raw Logs into a Structured DataFrame

This step converts unstructured log lines into tabular columns:
- `client_ip`
- `timestamp_raw`
- `method`, `endpoint`, `http_version`
- `status_code`, `bytes_sent`

We then create A/S-oriented fields such as `ticket_id`, `service_type`, and `event_text`.

In [4]:
log_pattern = re.compile(
    r'(?P<client_ip>\S+) \S+ \S+ \[(?P<timestamp_raw>[^\]]+)\] "(?P<method>\S+) (?P<endpoint>\S+) (?P<http_version>[^"]+)" (?P<status_code>\d{3}) (?P<bytes_sent>\S+)'
)

records = []
for line in raw_log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
    m = log_pattern.search(line)
    if not m:
        continue
    rec = m.groupdict()
    endpoint = rec["endpoint"]

    ticket_match = re.search(r"ticket=([A-Za-z0-9\-]+)", endpoint)
    service_match = re.search(r"/service/([A-Za-z0-9_\-]+)", endpoint)

    rec["ticket_id"] = ticket_match.group(1) if ticket_match else None
    rec["service_type_raw"] = service_match.group(1) if service_match else None
    rec["event_text"] = f"{rec['method']} {endpoint} -> {rec['status_code']}"
    records.append(rec)

df = pd.DataFrame(records)

if df.empty:
    raise ValueError("No rows parsed. Check regex or source file format.")

print("Parsed rows:", len(df))
df.head(8)

Parsed rows: 10006


,client_ip,timestamp_raw,method,endpoint,http_version,status_code,bytes_sent,ticket_id,service_type_raw,event_text
0,83.149.9.216,17/May/2015:10:05:03 +0000,GET,/presentations/logstash-monitorama-2013/images/kibana-search.png,HTTP/1.1,200,203023,NaN,NaN,GET /presentations/logstash-monitorama-2013/images/kibana-search.png -> 200
1,83.149.9.216,17/May/2015:10:05:43 +0000,GET,/presentations/logstash-monitorama-2013/images/kibana-dashboard3.png,HTTP/1.1,200,171717,NaN,NaN,GET /presentations/logstash-monitorama-2013/images/kibana-dashboard3.png -> 200
2,83.149.9.216,17/May/2015:10:05:47 +0000,GET,/presentations/logstash-monitorama-2013/plugin/highlight/highlight.js,HTTP/1.1,200,26185,NaN,NaN,GET /presentations/logstash-monitorama-2013/plugin/highlight/highlight.js -> 200
3,83.149.9.216,17/May/2015:10:05:12 +0000,GET,/presentations/logstash-monitorama-2013/plugin/zoom-js/zoom.js,HTTP/1.1,200,7697,NaN,NaN,GET /presentations/logstash-monitorama-2013/plugin/zoom-js/zoom.js -> 200
4,83.149.9.216,17/May/2015:10:05:07 +0000,GET,/presentations/logstash-monitorama-2013/plugin/notes/notes.js,HTTP/1.1,200,2892,NaN,NaN,GET /presentations/logstash-monitorama-2013/plugin/notes/notes.js -> 200
5,83.149.9.216,17/May/2015:10:05:34 +0000,GET,/presentations/logstash-monitorama-2013/images/sad-medic.png,HTTP/1.1,200,430406,NaN,NaN,GET /presentations/logstash-monitorama-2013/images/sad-medic.png -> 200
6,83.149.9.216,17/May/2015:10:05:57 +0000,GET,/presentations/logstash-monitorama-2013/css/fonts/Roboto-Bold.ttf,HTTP/1.1,200,38720,NaN,NaN,GET /presentations/logstash-monitorama-2013/css/fonts/Roboto-Bold.ttf -> 200
7,83.149.9.216,17/May/2015:10:05:50 +0000,GET,/presentations/logstash-monitorama-2013/css/fonts/Roboto-Regular.ttf,HTTP/1.1,200,41820,NaN,NaN,GET /presentations/logstash-monitorama-2013/css/fonts/Roboto-Regular.ttf -> 200


## 3) Data Type Filtering and Conversion

Now we convert columns to proper data types and filter invalid rows:
- Parse timestamps into `datetime`
- Convert status and bytes to numeric
- Remove impossible values
- Keep only rows with meaningful ticket/service information

In [5]:
work = df.copy()

# 1) Convert timestamp
work["timestamp"] = pd.to_datetime(
    work["timestamp_raw"],
    format="%d/%b/%Y:%H:%M:%S %z",
    errors="coerce",
)

# 2) Convert numeric columns
work["status_code"] = pd.to_numeric(work["status_code"], errors="coerce")
work["bytes_sent"] = pd.to_numeric(work["bytes_sent"].replace("-", pd.NA), errors="coerce")

# 3) Basic filtering rules
before = len(work)
work = work[work["timestamp"].notna()]
work = work[work["status_code"].between(100, 599, inclusive="both")]
work = work[work["service_type_raw"].notna()]
after = len(work)

print(f"Rows before filtering: {before}")
print(f"Rows after filtering:  {after}")
work[["timestamp", "status_code", "bytes_sent", "ticket_id", "service_type_raw"]].head(8)

Rows before filtering: 10006
Rows after filtering:  6


,timestamp,status_code,bytes_sent,ticket_id,service_type_raw
10000,2026-04-15 08:31:20+00:00,200,482.0,AS-1001,aircon
10001,2026-04-15 08:31:30+00:00,500,1210.0,AS-1002,washer
10002,2026-04-15 08:31:31+00:00,500,1210.0,AS-1002,washer
10003,2026-04-15 08:33:40+00:00,404,532.0,AS-1003,fridge
10004,2026-04-15 08:34:01+00:00,200,900.0,AS-1003,fridge
10005,2026-04-15 09:00:00+00:00,200,640.0,AS-1004,aircon


## 4) Cleansing and Normalization for A/S Logs

In this step we normalize business fields to make downstream analytics/ML reliable:
- Standardize `service_type`
- Add normalized `status_label`
- Build a canonical text field
- Clean string noise (trim spaces, lowercase where needed)

In [6]:
service_map = {
    "aircon": "air_conditioner",
    "ac": "air_conditioner",
    "washer": "washing_machine",
    "washingmachine": "washing_machine",
    "fridge": "refrigerator",
    "refrigerator": "refrigerator",
}

clean = work.copy()

clean["service_type"] = (
    clean["service_type_raw"].astype(str).str.strip().str.lower().map(service_map).fillna("other")
)

clean["ticket_id"] = clean["ticket_id"].fillna("UNKNOWN").astype(str).str.strip().str.upper()

clean["status_label"] = pd.cut(
    clean["status_code"],
    bins=[99, 299, 399, 499, 599],
    labels=["success", "redirect", "client_error", "server_error"],
)

clean["event_text_normalized"] = (
    "ticket=" + clean["ticket_id"]
    + " | service=" + clean["service_type"]
    + " | status=" + clean["status_label"].astype(str)
)

clean = clean.sort_values("timestamp").reset_index(drop=True)

clean[[
    "timestamp", "ticket_id", "service_type", "status_code", "status_label", "event_text_normalized"
]].head(10)

,timestamp,ticket_id,service_type,status_code,status_label,event_text_normalized
0,2026-04-15 08:31:20+00:00,AS-1001,air_conditioner,200,success,ticket=AS-1001 | service=air_conditioner | status=success
1,2026-04-15 08:31:30+00:00,AS-1002,washing_machine,500,server_error,ticket=AS-1002 | service=washing_machine | status=server_error
2,2026-04-15 08:31:31+00:00,AS-1002,washing_machine,500,server_error,ticket=AS-1002 | service=washing_machine | status=server_error
3,2026-04-15 08:33:40+00:00,AS-1003,refrigerator,404,client_error,ticket=AS-1003 | service=refrigerator | status=client_error
4,2026-04-15 08:34:01+00:00,AS-1003,refrigerator,200,success,ticket=AS-1003 | service=refrigerator | status=success
5,2026-04-15 09:00:00+00:00,AS-1004,air_conditioner,200,success,ticket=AS-1004 | service=air_conditioner | status=success


## 5) Create CSV and DOCX Inputs (for Format Conversion Practice)

To practice conversion pipelines, we intentionally create two source formats:
- `as_logs.csv`
- `as_logs.docx`

The DOCX file stores each event as a single line of key-value text.

In [7]:
csv_path = INTERIM_DIR / "as_logs.csv"
docx_path = INTERIM_DIR / "as_logs.docx"

csv_cols = [
    "timestamp", "ticket_id", "service_type", "status_code", "status_label", "client_ip", "event_text_normalized"
]
clean[csv_cols].to_csv(csv_path, index=False, encoding="utf-8")

doc = Document()
doc.add_heading("A/S Service Log Notes", level=1)
for _, row in clean.head(200).iterrows():
    line = (
        f"timestamp={row['timestamp']} | ticket_id={row['ticket_id']} | service_type={row['service_type']} | "
        f"status_code={int(row['status_code'])} | status_label={row['status_label']} | "
        f"client_ip={row['client_ip']} | event_text={row['event_text_normalized']}"
    )
    doc.add_paragraph(line)
doc.save(docx_path)

print("Created:")
print("-", csv_path)
print("-", docx_path)

Created:
- /home/ethan/newgen/KMOU_Course/Module_4/data/interim/as_logs.csv
- /home/ethan/newgen/KMOU_Course/Module_4/data/interim/as_logs.docx


## 6) Convert CSV to JSONL

JSONL (JSON Lines) is ideal for ML pipelines because each line is a standalone JSON record.

Here we read the CSV and write one JSON object per row.

In [8]:
jsonl_from_csv = PROCESSED_DIR / "as_logs_from_csv.jsonl"

csv_df = pd.read_csv(csv_path)

with jsonl_from_csv.open("w", encoding="utf-8") as f:
    for rec in csv_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved:", jsonl_from_csv)
print("Records:", len(csv_df))
print("Preview:")
for i, line in enumerate(jsonl_from_csv.read_text(encoding="utf-8").splitlines()[:2], start=1):
    print(i, line)

Saved: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_from_csv.jsonl
Records: 6
Preview:
1 {"timestamp": "2026-04-15 08:31:20+00:00", "ticket_id": "AS-1001", "service_type": "air_conditioner", "status_code": 200, "status_label": "success", "client_ip": "203.0.113.10", "event_text_normalized": "ticket=AS-1001 | service=air_conditioner | status=success"}
2 {"timestamp": "2026-04-15 08:31:30+00:00", "ticket_id": "AS-1002", "service_type": "washing_machine", "status_code": 500, "status_label": "server_error", "client_ip": "198.51.100.77", "event_text_normalized": "ticket=AS-1002 | service=washing_machine | status=server_error"}


## 7) Convert DOCX to JSONL

Now we parse DOCX paragraphs that follow this schema:
`key=value | key=value | ...`

Then we serialize them to JSONL as well.

In [9]:
jsonl_from_docx = PROCESSED_DIR / "as_logs_from_docx.jsonl"

doc = Document(docx_path)
doc_records = []

for para in doc.paragraphs:
    text = para.text.strip()
    if not text or "=" not in text:
        continue
    parts = [p.strip() for p in text.split("|")]
    rec = {}
    for part in parts:
        if "=" not in part:
            continue
        k, v = part.split("=", 1)
        rec[k.strip()] = v.strip()
    if rec:
        doc_records.append(rec)

with jsonl_from_docx.open("w", encoding="utf-8") as f:
    for rec in doc_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved:", jsonl_from_docx)
print("Records:", len(doc_records))
print("Preview:")
for i, line in enumerate(jsonl_from_docx.read_text(encoding="utf-8").splitlines()[:2], start=1):
    print(i, line)

Saved: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_from_docx.jsonl
Records: 6
Preview:
1 {"timestamp": "2026-04-15 08:31:20+00:00", "ticket_id": "AS-1001", "service_type": "air_conditioner", "status_code": "200", "status_label": "success", "client_ip": "203.0.113.10", "event_text": "ticket=AS-1001", "service": "air_conditioner", "status": "success"}
2 {"timestamp": "2026-04-15 08:31:30+00:00", "ticket_id": "AS-1002", "service_type": "washing_machine", "status_code": "500", "status_label": "server_error", "client_ip": "198.51.100.77", "event_text": "ticket=AS-1002", "service": "washing_machine", "status": "server_error"}


## 8) Merge JSONL Sources and Normalize Final Schema

Real pipelines often ingest multiple source formats. We now:
- Read both JSONL files
- Align field names
- Normalize data types
- Create a unified dataset

In [10]:
def read_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows

rows_csv = read_jsonl(jsonl_from_csv)
rows_docx = read_jsonl(jsonl_from_docx)

all_rows = rows_csv + rows_docx
unified = pd.DataFrame(all_rows)

# unify a few expected columns
if "event_text" in unified.columns and "event_text_normalized" not in unified.columns:
    unified["event_text_normalized"] = unified["event_text"]

unified["timestamp"] = pd.to_datetime(unified["timestamp"], errors="coerce", utc=True)
unified["status_code"] = pd.to_numeric(unified["status_code"], errors="coerce")
unified["ticket_id"] = unified["ticket_id"].astype(str).str.upper().str.strip()
unified["service_type"] = unified["service_type"].astype(str).str.lower().str.strip()

unified = unified.dropna(subset=["timestamp", "status_code"]).reset_index(drop=True)

print("Unified rows:", len(unified))
unified.head(6)

Unified rows: 12


,timestamp,ticket_id,service_type,status_code,status_label,client_ip,event_text_normalized,event_text,service,status
0,2026-04-15 08:31:20+00:00,AS-1001,air_conditioner,200,success,203.0.113.10,ticket=AS-1001 | service=air_conditioner | status=success,NaN,NaN,NaN
1,2026-04-15 08:31:30+00:00,AS-1002,washing_machine,500,server_error,198.51.100.77,ticket=AS-1002 | service=washing_machine | status=server_error,NaN,NaN,NaN
2,2026-04-15 08:31:31+00:00,AS-1002,washing_machine,500,server_error,198.51.100.77,ticket=AS-1002 | service=washing_machine | status=server_error,NaN,NaN,NaN
3,2026-04-15 08:33:40+00:00,AS-1003,refrigerator,404,client_error,192.0.2.20,ticket=AS-1003 | service=refrigerator | status=client_error,NaN,NaN,NaN
4,2026-04-15 08:34:01+00:00,AS-1003,refrigerator,200,success,192.0.2.20,ticket=AS-1003 | service=refrigerator | status=success,NaN,NaN,NaN
5,2026-04-15 09:00:00+00:00,AS-1004,air_conditioner,200,success,198.51.100.90,ticket=AS-1004 | service=air_conditioner | status=success,NaN,NaN,NaN


## 9) Deduplication Techniques

We apply two styles:

1. **Exact duplicate removal**
   - same ticket, timestamp, status, and event text

2. **Near duplicate removal**
   - based on a normalized key built from selected columns
   - useful when spacing/casing differ but semantic content is the same

In [11]:
dedup = unified.copy()
before = len(dedup)

dedup = dedup.drop_duplicates(
    subset=["ticket_id", "timestamp", "status_code", "event_text_normalized"],
    keep="first",
)
after_exact = len(dedup)

def normalize_text_for_key(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9_=|\- ]", "", text)
    return text

dedup["near_dup_key"] = (
    dedup["ticket_id"].astype(str).str.upper().str.strip()
    + "||"
    + dedup["service_type"].astype(str).str.lower().str.strip()
    + "||"
    + dedup["event_text_normalized"].astype(str).apply(normalize_text_for_key)
)

dedup = dedup.drop_duplicates(subset=["near_dup_key"], keep="first").drop(columns=["near_dup_key"])
after_near = len(dedup)

print(f"Rows before dedup:         {before}")
print(f"After exact dedup:         {after_exact}")
print(f"After near-duplicate dedup:{after_near}")

Rows before dedup:         12
After exact dedup:         12
After near-duplicate dedup:9


## 10) PII Detection and Masking

In support logs, common PII can appear in free text:
- Email
- Phone numbers
- IP addresses

We detect and mask these values before exporting.

In [12]:
# Add synthetic PII examples to demonstrate masking logic.
if len(dedup) > 0:
    dedup.loc[dedup.index[0], "event_text_normalized"] += " | contact=jane.doe@example.com | phone=010-1234-5678"
if len(dedup) > 1:
    dedup.loc[dedup.index[1], "event_text_normalized"] += " | alt_phone=+82 10 9999 1111"

email_re = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
phone_re = re.compile(r"(?:\+?\d{1,3}[\s-]?)?(?:\d{2,4}[\s-]?){2,4}\d{3,4}")
ip_re = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

def mask_pii(text: str) -> str:
    x = str(text)
    x = email_re.sub("[EMAIL_REDACTED]", x)
    x = phone_re.sub("[PHONE_REDACTED]", x)
    x = ip_re.sub("[IP_REDACTED]", x)
    return x

pii = dedup.copy()
pii["event_text_masked"] = pii["event_text_normalized"].apply(mask_pii)
pii["client_ip_masked"] = pii["client_ip"].astype(str).apply(mask_pii)

pii[["event_text_normalized", "event_text_masked", "client_ip", "client_ip_masked"]].head(5)

,event_text_normalized,event_text_masked,client_ip,client_ip_masked
0,ticket=AS-1001 | service=air_conditioner | status=success | contact=jane.doe@example.com | phone=010-1234-5678,ticket=AS-1001 | service=air_conditioner | status=success | contact=[EMAIL_REDACTED] | phone=[PHONE_REDACTED],203.0.113.10,[IP_REDACTED]
1,ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=+82 10 9999 1111,ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=[PHONE_REDACTED],198.51.100.77,[IP_REDACTED]
3,ticket=AS-1003 | service=refrigerator | status=client_error,ticket=AS-1003 | service=refrigerator | status=client_error,192.0.2.20,[IP_REDACTED]
4,ticket=AS-1003 | service=refrigerator | status=success,ticket=AS-1003 | service=refrigerator | status=success,192.0.2.20,[IP_REDACTED]
5,ticket=AS-1004 | service=air_conditioner | status=success,ticket=AS-1004 | service=air_conditioner | status=success,198.51.100.90,[IP_REDACTED]


## 11) Final Export to Training-Ready JSONL

We now export a clean, normalized, deduplicated, and masked dataset.

Recommended final fields:
- `timestamp`
- `ticket_id`
- `service_type`
- `status_code`
- `status_label`
- `event_text_masked`
- `client_ip_masked`

In [13]:
final_cols = [
    "timestamp", "ticket_id", "service_type", "status_code", "status_label", "event_text_masked", "client_ip_masked"
]

final_df = pii.copy()
final_df["timestamp"] = final_df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
final_df = final_df[final_cols].reset_index(drop=True)

final_jsonl_path = PROCESSED_DIR / "as_logs_final_cleaned.jsonl"
with final_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in final_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Final JSONL saved:", final_jsonl_path)
print("Final record count:", len(final_df))
print("Sample output lines:")
for line in final_jsonl_path.read_text(encoding="utf-8").splitlines()[:3]:
    print(line)

Final JSONL saved: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_final_cleaned.jsonl
Final record count: 9
Sample output lines:
{"timestamp": "2026-04-15T08:31:20Z", "ticket_id": "AS-1001", "service_type": "air_conditioner", "status_code": 200, "status_label": "success", "event_text_masked": "ticket=AS-1001 | service=air_conditioner | status=success | contact=[EMAIL_REDACTED] | phone=[PHONE_REDACTED]", "client_ip_masked": "[IP_REDACTED]"}
{"timestamp": "2026-04-15T08:31:30Z", "ticket_id": "AS-1002", "service_type": "washing_machine", "status_code": 500, "status_label": "server_error", "event_text_masked": "ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=[PHONE_REDACTED]", "client_ip_masked": "[IP_REDACTED]"}
{"timestamp": "2026-04-15T08:33:40Z", "ticket_id": "AS-1003", "service_type": "refrigerator", "status_code": 404, "status_label": "client_error", "event_text_masked": "ticket=AS-1003 | service=refrigerator | status=client_error", "client_

## 12) Quality Checklist (What Learners Should Verify)

After running all cells, verify:

- [ ] No invalid timestamps in final output
- [ ] `status_code` is numeric and in valid HTTP range
- [ ] Duplicate records are reduced
- [ ] PII patterns are redacted
- [ ] Output is valid JSONL (1 JSON object per line)

Optional extension ideas:
1. Add hash-based deterministic IDs per record.
2. Add language detection or text normalization for multilingual logs.
3. Add unit tests for parser, dedup logic, and PII regexes.

## 13) Advanced Practice Pack: More Cases, More Methods

This section expands the lesson with additional real-world data quality issues.

New scenarios:
1. Missing values and schema drift
2. Inconsistent categorical values (e.g., service aliases)
3. Conflicting records for the same ticket
4. Outlier detection for numeric fields
5. Timestamp standardization across mixed formats
6. Extra format conversion example (XML -> JSONL)

The goal is to compare **multiple cleansing strategies** and understand trade-offs.

In [14]:
# Build a deliberately messy A/S dataset for advanced practice.
advanced_raw = pd.DataFrame(
    [
        {
            "ticket_id": "as-2001",
            "service_type": "AirCon",
            "status_code": "200",
            "bytes_sent": "820",
            "timestamp": "2026-04-16 10:20:30+09:00",
            "agent_note": "Resolved. customer email: alpha.user@example.com",
            "source": "crm_csv",
        },
        {
            "ticket_id": "AS-2001",
            "service_type": "air conditioner",
            "status_code": 200,
            "bytes_sent": 820,
            "timestamp": "16/04/2026 10:20:30",
            "agent_note": "resolved",
            "source": "docx_export",
        },
        {
            "ticket_id": "AS-2002",
            "service_type": "washer",
            "status_code": "500",
            "bytes_sent": "9999999",
            "timestamp": "2026/04/16 10:22",
            "agent_note": "Tech requested callback 010-1111-2222",
            "source": "crm_csv",
        },
        {
            "ticket_id": "AS-2003",
            "service_type": "fridge",
            "status_code": "404",
            "bytes_sent": "-",
            "timestamp": "2026-04-16T01:00:00Z",
            "agent_note": "Part not found",
            "source": "web_form",
        },
        {
            "ticket_id": None,
            "service_type": "AC",
            "status_code": "200",
            "bytes_sent": "500",
            "timestamp": "bad_timestamp",
            "agent_note": "Missing ticket id",
            "source": "legacy",
        },
        {
            "ticket_id": "AS-2004",
            "service_type": None,
            "status_code": "301",
            "bytes_sent": "620",
            "timestamp": "2026-04-16 10:28:00+09:00",
            "agent_note": "redirected",
            "source": "legacy",
        },
        {
            "ticket_id": "AS-2005",
            "service_type": "washingmachine",
            "status_code": "two hundred",
            "bytes_sent": "700",
            "timestamp": "2026-04-16 10:30:00+09:00",
            "agent_note": "manual correction needed",
            "source": "manual_entry",
        },
    ]
)

print("Advanced raw sample:")
display(advanced_raw)

Advanced raw sample:


,ticket_id,service_type,status_code,bytes_sent,timestamp,agent_note,source
0,as-2001,AirCon,200,820,2026-04-16 10:20:30+09:00,Resolved. customer email: alpha.user@example.com,crm_csv
1,AS-2001,air conditioner,200,820,16/04/2026 10:20:30,resolved,docx_export
2,AS-2002,washer,500,9999999,2026/04/16 10:22,Tech requested callback 010-1111-2222,crm_csv
3,AS-2003,fridge,404,-,2026-04-16T01:00:00Z,Part not found,web_form
4,NaN,AC,200,500,bad_timestamp,Missing ticket id,legacy
5,AS-2004,NaN,301,620,2026-04-16 10:28:00+09:00,redirected,legacy
6,AS-2005,washingmachine,two hundred,700,2026-04-16 10:30:00+09:00,manual correction needed,manual_entry


## 14) Method A vs Method B for Missing Values

There is no single "best" method. We compare two common approaches:

- **Method A (strict):** drop rows with missing critical keys (`ticket_id`, `service_type`, `timestamp`)
- **Method B (impute/fill):** keep rows by imputing placeholders and using parsing fallback

Use this to teach the trade-off between **data quality purity** and **data retention**.

In [15]:
# Common pre-processing
adv = advanced_raw.copy()
adv["ticket_id"] = adv["ticket_id"].astype(str).str.upper().str.strip()
adv["ticket_id"] = adv["ticket_id"].replace({"NONE": pd.NA, "NAN": pd.NA})

# Flexible timestamp parser for mixed formats
adv["timestamp_parsed"] = pd.to_datetime(adv["timestamp"], errors="coerce", utc=True)
adv["status_code_num"] = pd.to_numeric(adv["status_code"], errors="coerce")
adv["bytes_sent_num"] = pd.to_numeric(adv["bytes_sent"].replace("-", pd.NA), errors="coerce")

# Method A: strict drop
method_a = adv.dropna(subset=["ticket_id", "service_type", "timestamp_parsed"]).copy()

# Method B: impute and keep
method_b = adv.copy()
method_b["ticket_id"] = method_b["ticket_id"].fillna("UNKNOWN_TICKET")
method_b["service_type"] = method_b["service_type"].fillna("unknown_service")
method_b["timestamp_parsed"] = method_b["timestamp_parsed"].fillna(pd.Timestamp("1970-01-01", tz="UTC"))

print("Method A row count (strict):", len(method_a))
print("Method B row count (impute):", len(method_b))

comparison = pd.DataFrame(
    {
        "metric": ["rows_kept", "missing_ticket_after", "missing_service_after", "missing_ts_after"],
        "method_a_strict": [
            len(method_a),
            method_a["ticket_id"].isna().sum(),
            method_a["service_type"].isna().sum(),
            method_a["timestamp_parsed"].isna().sum(),
        ],
        "method_b_impute": [
            len(method_b),
            method_b["ticket_id"].isna().sum(),
            method_b["service_type"].isna().sum(),
            method_b["timestamp_parsed"].isna().sum(),
        ],
    }
)
display(comparison)

Method A row count (strict): 2
Method B row count (impute): 7


,metric,method_a_strict,method_b_impute
0,rows_kept,2,7
1,missing_ticket_after,0,0
2,missing_service_after,0,0
3,missing_ts_after,0,0


## 15) Categorical Normalization Methods

This section demonstrates two normalization techniques for category columns:

1. **Dictionary mapping** (fast, controlled)
2. **Rule-based normalization** using regex and token cleanup (more flexible)

You can compare outputs and decide which method is easier to maintain.

In [16]:
cat_df = method_b.copy()

# Method 1: dictionary mapping
service_map_dict = {
    "aircon": "air_conditioner",
    "ac": "air_conditioner",
    "air conditioner": "air_conditioner",
    "washer": "washing_machine",
    "washingmachine": "washing_machine",
    "fridge": "refrigerator",
}

cat_df["service_dict_norm"] = (
    cat_df["service_type"].astype(str).str.lower().str.strip().map(service_map_dict).fillna("other")
)

# Method 2: rule-based normalization

def normalize_service_rule(x: str) -> str:
    t = str(x).lower().strip()
    t = re.sub(r"[^a-z ]", "", t)
    if re.search(r"\b(ac|aircon|air conditioner)\b", t):
        return "air_conditioner"
    if re.search(r"\b(washer|washing machine|washingmachine)\b", t):
        return "washing_machine"
    if re.search(r"\b(fridge|refrigerator)\b", t):
        return "refrigerator"
    if "unknown" in t:
        return "unknown_service"
    return "other"

cat_df["service_rule_norm"] = cat_df["service_type"].apply(normalize_service_rule)

print("Comparison of category normalization methods:")
display(cat_df[["service_type", "service_dict_norm", "service_rule_norm"]])

Comparison of category normalization methods:


,service_type,service_dict_norm,service_rule_norm
0,AirCon,air_conditioner,air_conditioner
1,air conditioner,air_conditioner,air_conditioner
2,washer,washing_machine,washing_machine
3,fridge,refrigerator,refrigerator
4,AC,air_conditioner,air_conditioner
5,unknown_service,other,unknown_service
6,washingmachine,washing_machine,washing_machine


## 16) Conflict Resolution for Same Ticket

Sometimes the same ticket appears multiple times with different statuses from different systems.

Two common rules:
- **Latest-event wins** (based on timestamp)
- **Priority-based wins** (e.g., server_error > client_error > success)

This teaches that data cleansing can include **business rules**, not just technical cleaning.

In [17]:
conflict_df = cat_df.copy()
conflict_df["status_code_num"] = pd.to_numeric(conflict_df["status_code_num"], errors="coerce")
conflict_df["status_label"] = pd.cut(
    conflict_df["status_code_num"],
    bins=[99, 299, 399, 499, 599],
    labels=["success", "redirect", "client_error", "server_error"],
)

# Rule 1: latest timestamp wins
latest_wins = (
    conflict_df.sort_values("timestamp_parsed")
    .drop_duplicates(subset=["ticket_id"], keep="last")
    [["ticket_id", "timestamp_parsed", "status_code_num", "status_label", "source"]]
    .sort_values("ticket_id")
)

# Rule 2: priority-based wins
priority_map = {"server_error": 4, "client_error": 3, "redirect": 2, "success": 1}
prio_df = conflict_df.copy()
prio_df["priority"] = prio_df["status_label"].astype(str).map(priority_map).fillna(0)
priority_wins = (
    prio_df.sort_values(["ticket_id", "priority", "timestamp_parsed"], ascending=[True, False, False])
    .drop_duplicates(subset=["ticket_id"], keep="first")
    [["ticket_id", "timestamp_parsed", "status_code_num", "status_label", "source", "priority"]]
    .sort_values("ticket_id")
)

print("Latest-event wins:")
display(latest_wins)
print("Priority-based wins:")
display(priority_wins)

Latest-event wins:


,ticket_id,timestamp_parsed,status_code_num,status_label,source
0,AS-2001,2026-04-16 01:20:30+00:00,200.0,success,crm_csv
2,AS-2002,1970-01-01 00:00:00+00:00,500.0,server_error,crm_csv
3,AS-2003,1970-01-01 00:00:00+00:00,404.0,client_error,web_form
5,AS-2004,2026-04-16 01:28:00+00:00,301.0,redirect,legacy
6,AS-2005,2026-04-16 01:30:00+00:00,NaN,NaN,manual_entry
4,UNKNOWN_TICKET,1970-01-01 00:00:00+00:00,200.0,success,legacy


Priority-based wins:


,ticket_id,timestamp_parsed,status_code_num,status_label,source,priority
0,AS-2001,2026-04-16 01:20:30+00:00,200.0,success,crm_csv,1.0
2,AS-2002,1970-01-01 00:00:00+00:00,500.0,server_error,crm_csv,4.0
3,AS-2003,1970-01-01 00:00:00+00:00,404.0,client_error,web_form,3.0
5,AS-2004,2026-04-16 01:28:00+00:00,301.0,redirect,legacy,2.0
6,AS-2005,2026-04-16 01:30:00+00:00,NaN,NaN,manual_entry,0.0
4,UNKNOWN_TICKET,1970-01-01 00:00:00+00:00,200.0,success,legacy,1.0


## 17) Outlier Handling for Numeric Columns

For fields like `bytes_sent`, we demonstrate two practical methods:

- **IQR capping (winsorization style)**
- **Z-score filtering**

Learners should inspect how each method changes distribution and retained rows.

In [18]:
num_df = conflict_df.copy()
num_df = num_df[num_df["bytes_sent_num"].notna()].copy()

# Method 1: IQR capping
q1 = num_df["bytes_sent_num"].quantile(0.25)
q3 = num_df["bytes_sent_num"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
num_df["bytes_iqr_capped"] = num_df["bytes_sent_num"].clip(lower=lower, upper=upper)

# Method 2: Z-score filtering
mean_val = num_df["bytes_sent_num"].mean()
std_val = num_df["bytes_sent_num"].std(ddof=0)
if std_val == 0:
    num_df["z_score"] = 0.0
else:
    num_df["z_score"] = (num_df["bytes_sent_num"] - mean_val) / std_val

z_filtered = num_df[num_df["z_score"].abs() <= 3].copy()

print("IQR bounds:", (lower, upper))
print("Rows before z-filter:", len(num_df))
print("Rows after z-filter:", len(z_filtered))

display(num_df[["ticket_id", "bytes_sent_num", "bytes_iqr_capped", "z_score"]])

IQR bounds: (np.float64(370.0), np.float64(1090.0))
Rows before z-filter: 6
Rows after z-filter: 6


,ticket_id,bytes_sent_num,bytes_iqr_capped,z_score
0,AS-2001,820.0,820.0,-0.447179
1,AS-2001,820.0,820.0,-0.447179
2,AS-2002,9999999.0,1090.0,2.236068
4,UNKNOWN_TICKET,500.0,500.0,-0.447265
5,AS-2004,620.0,620.0,-0.447233
6,AS-2005,700.0,700.0,-0.447211


## 18) Extra Conversion Example: XML -> JSONL

To enrich conversion practice beyond CSV/DOCX, this example converts XML event logs to JSONL.

This is useful when integrating with legacy systems or middleware exports.

In [19]:
import xml.etree.ElementTree as ET

xml_path = INTERIM_DIR / "as_logs.xml"
xml_jsonl_path = PROCESSED_DIR / "as_logs_from_xml.jsonl"

xml_text = """<logs>
  <log><ticket_id>AS-3001</ticket_id><service_type>ac</service_type><status_code>200</status_code><note>ok</note></log>
  <log><ticket_id>AS-3002</ticket_id><service_type>washer</service_type><status_code>500</status_code><note>call +82-10-5555-6666</note></log>
  <log><ticket_id>AS-3003</ticket_id><service_type>fridge</service_type><status_code>404</status_code><note>email support@sample.com</note></log>
</logs>"""
xml_path.write_text(xml_text, encoding="utf-8")

root = ET.fromstring(xml_text)
xml_records = []
for log in root.findall("log"):
    st = pd.to_numeric(log.findtext("status_code"), errors="coerce")
    xml_records.append(
        {
            "ticket_id": (log.findtext("ticket_id") or "").strip(),
            "service_type": (log.findtext("service_type") or "").strip(),
            "status_code": None if pd.isna(st) else int(st),
            "note": (log.findtext("note") or "").strip(),
            "source": "xml",
        }
    )

with xml_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in xml_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved XML source:", xml_path)
print("Saved JSONL from XML:", xml_jsonl_path)
print("Records:", len(xml_records))

Saved XML source: /home/ethan/newgen/KMOU_Course/Module_4/data/interim/as_logs.xml
Saved JSONL from XML: /home/ethan/newgen/KMOU_Course/Module_4/data/processed/as_logs_from_xml.jsonl
Records: 3


## 19) Data Quality Scoring (Rule-based)

A simple quality score helps learners evaluate output quality quantitatively.

Example scoring logic (0 to 100):
- +25 if ticket format is valid
- +25 if timestamp is parseable
- +25 if status code is valid HTTP range
- +25 if no obvious PII remains in note/text

This makes data cleansing outcomes measurable and easy to compare across methods.

In [20]:
score_df = pii.copy()

ticket_ok_re = re.compile(r"^AS-\d{4,}$")

# Reuse PII patterns from earlier section if present.
if "email_re" not in globals():
    email_re = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
if "phone_re" not in globals():
    phone_re = re.compile(r"(?:\+?\d{1,3}[\s-]?)?(?:\d{2,4}[\s-]?){2,4}\d{3,4}")


def quality_score_row(row):
    score = 0

    ticket = str(row.get("ticket_id", ""))
    ts = row.get("timestamp")
    status = row.get("status_code")
    text = str(row.get("event_text_masked", row.get("event_text_normalized", "")))

    if ticket_ok_re.match(ticket):
        score += 25
    if pd.notna(ts):
        score += 25
    if pd.notna(status) and 100 <= float(status) <= 599:
        score += 25

    has_pii = bool(email_re.search(text) or phone_re.search(text))
    if not has_pii:
        score += 25

    return score

score_df["quality_score"] = score_df.apply(quality_score_row, axis=1)

print("Quality score distribution:")
display(score_df["quality_score"].value_counts().sort_index())
print("Average score:", round(score_df["quality_score"].mean(), 2))
display(score_df[["ticket_id", "status_code", "event_text_masked", "quality_score"]].head(10))

Quality score distribution:


quality_score
100    9
Name: count, dtype: int64

Average score: 100.0


,ticket_id,status_code,event_text_masked,quality_score
0,AS-1001,200,ticket=AS-1001 | service=air_conditioner | status=success | contact=[EMAIL_REDACTED] | phone=[PHONE_REDACTED],100
1,AS-1002,500,ticket=AS-1002 | service=washing_machine | status=server_error | alt_phone=[PHONE_REDACTED],100
3,AS-1003,404,ticket=AS-1003 | service=refrigerator | status=client_error,100
4,AS-1003,200,ticket=AS-1003 | service=refrigerator | status=success,100
5,AS-1004,200,ticket=AS-1004 | service=air_conditioner | status=success,100
6,AS-1001,200,nan,100
7,AS-1002,500,nan,100
9,AS-1003,404,nan,100
11,AS-1004,200,nan,100


## 20) Suggested Learner Exercises (Hands-on)

1. Add at least 10 new noisy records (missing fields, typo categories, invalid status code, mixed timezone).
2. Compare strict vs imputation strategy and explain when each is preferred.
3. Add one more conversion path: `TSV -> JSONL`.
4. Upgrade near-duplicate logic with similarity matching (e.g., `difflib.SequenceMatcher`).
5. Expand PII masking for resident IDs or credit-card-like patterns.
6. Build a mini benchmark table: rows kept, duplicates removed, PII leaks found, quality score average.

These exercises make the workshop richer and closer to real production cleansing pipelines.